# YOLO — Session 1: Object Detection
### ITI · Instructor: Ahmed Abdelsalam

You've used pretrained models to answer *"what is in this image?"*. **Object detection** answers a bigger question: *"what is in this image, and **where**?"* — drawing a box around every object.

**YOLO** ("You Only Look Once") is the most popular detector: fast enough for real-time video, and incredibly easy to use.

**Today's map**
1. Classification vs detection
2. What a detection actually is (box + label + confidence)
3. How YOLO works (the intuition)
4. Running a pretrained YOLO
5. Reading the results
6. Filtering and counting objects
7. Detection on video (the idea)

> **Setup:** made for **Google Colab**. A GPU runtime helps but isn't required: *Runtime → Change runtime type → GPU*.

## Setup — install & import

Ultralytics is the library that gives us YOLO. One line to install.

In [1]:
!pip install ultralytics -q

from ultralytics import YOLO
import matplotlib.pyplot as plt
from skimage import data
from PIL import Image
print("Ultralytics ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics ready.


## 1. Classification vs detection

| | Classification | Detection |
|---|---|---|
| Question | *What* is this? | *What* and *where*? |
| Output | one label | many boxes, each with a label |
| Example | "this image has a cat" | "cat at [x1,y1,x2,y2], dog at [...]" |

Detection = finding **multiple** objects **and their locations** at once.

## 2. What a detection is

Every object YOLO finds comes with three things:

- **Bounding box** — four numbers `(x1, y1, x2, y2)`: the top-left and bottom-right corners of the box, in pixels.
- **Class** — what the object is (e.g. "person", "car").
- **Confidence** — how sure the model is, from 0 to 1 (e.g. 0.92 = 92% sure).

That's it. A detector's whole job is to output a list of `(box, class, confidence)`.

## 3. How YOLO works (the intuition)

The name says it: **You Only Look Once**. Older detectors scanned an image many times over. YOLO looks at the **whole image a single time** and predicts all boxes at once — which is why it's fast enough for live video.

The rough idea (no math needed today):
1. YOLO divides the image into a **grid**.
2. Each grid cell predicts boxes and class scores for objects near it.
3. It ends up with **many overlapping boxes**, so a cleanup step called **NMS** (Non-Max Suppression) keeps only the best box per object and removes duplicates.

Two words worth knowing:
- **IoU** (Intersection over Union) — how much two boxes overlap (0 = no overlap, 1 = identical). Used to decide if boxes refer to the same object.
- **NMS** — uses IoU to drop duplicate boxes, keeping the most confident one.

> You won't compute these by hand — Ultralytics does it all. Just know the vocabulary.

## 4. Running a pretrained YOLO

Ultralytics ships YOLO models already trained on **COCO** — a dataset of 80 everyday object classes (person, car, dog, cup, laptop, ...). Loading and running is two lines.

Model sizes go from `n` (nano, fastest) to `x` (extra-large, most accurate): `yolo11n`, `yolo11s`, `yolo11m`, `yolo11l`, `yolo11x`. We'll use **nano**.

In [2]:
# save a sample image to disk to run on
Image.fromarray(data.coffee()).save("scene.jpg")

model = YOLO("yolo11n.pt")            # downloads the nano model once
results = model2("scene.jpg", conf=0.5)          # run detection

# YOLO can draw the boxes for us:
annotated = results[0].plot()         # returns an image with boxes (in BGR)
plt.figure(figsize=(8, 6))
plt.imshow(annotated[:, :, ::-1])     # BGR -> RGB for display
plt.axis("off"); plt.title("YOLO detections"); plt.show()

NameError: name 'model2' is not defined

> Two lines and it found the objects, labelled them, and drew the boxes. That's the appeal of YOLO.

## 5. Reading the results

The annotated picture is nice, but usually you need the **numbers** — to count things, trigger alerts, etc. The detections live in `results[0].boxes`.

In [ ]:
result = results[0]
print("Number of objects found:", len(result.boxes))
print()

for box in result.boxes:
    class_id = int(box.cls)                 # numeric class id
    label = model.names[class_id]           # class name
    confidence = float(box.conf)            # 0..1
    x1, y1, x2, y2 = [round(v) for v in box.xyxy[0].tolist()]   # corner coords
    print(f"{label:15s} conf={confidence:.2f}  box=({x1},{y1})-({x2},{y2})")

- `box.cls` → class id, turned into a name with `model.names`
- `box.conf` → confidence
- `box.xyxy` → the four corner coordinates

🔵 **Your turn:** run the model on a different image — save `data.astronaut()` as `person.jpg`, detect, and print each object's label and confidence.

In [ ]:
# Write your code here
Image.fromarray(data.astronaut()).save("person.jpg")
model2 = YOLO("yolo11l.pt")
results = model2("person.jpg", conf=0.5)
result = results[0]
print("Number of objects found:", len(result.boxes))
plt.figure(figsize=(8, 6))
plt.imshow(result.plot()[:, :, ::-1])
plt.axis("off"); plt.title("YOLO detections"); plt.show()


## 6. Filtering and counting

Real applications ask questions like *"how many people?"* or *"only show detections above 80% confidence."* You just loop and filter.

In [ ]:
# Keep only confident detections
model3 = YOLO("yolo11x.pt")
result = model3("scene.jpg")[0]

CONF_THRESHOLD = 0.7
confident = [b for b in result.boxes if float(b.conf) >= CONF_THRESHOLD]
print(f"Objects above {CONF_THRESHOLD}: {len(confident)}")

# Count how many of each class
counts = {}
for box in result.boxes:
    label = model.names[int(box.cls)]
    counts[label] = counts.get(label, 0) + 1
print("Counts by class:", counts)

You can also ask YOLO to detect **only certain classes** with the `classes` argument (COCO class 0 = person).

In [ ]:
print(model.names)

In [ ]:
# detect only people (class 0)
people = model3("scene.jpg", classes=[41], verbose=False)[0]
print("Cup found:", len(people.boxes))
plt.figure(figsize=(8, 6))
plt.imshow(people.plot()[:, :, ::-1])
plt.axis("off"); plt.title("YOLO detections"); plt.show()

🔵 **Your turn:** using the `counts` idea, write code that prints how many `cup` objects are in `scene.jpg`.

In [ ]:
# Write your code here
cups = model2("scene.jpg", classes=[41],conf=0.7)[0]
print("Cup found:", len(cups.boxes))

## 7. Detection on video (the idea)

Because YOLO is fast, the exact same call works on **video** and **webcams** — YOLO just runs on each frame. On a file you'd write:

```python
# runs detection on every frame and saves an annotated video
model.predict("my_video.mp4", save=True)
```

Ultralytics also supports streams and webcams (`source=0`). We won't run video in class (it's slow without a GPU), but know that **nothing changes except the source** — image, folder of images, video, or camera.

In [ ]:
!pip install yt-dlp
!yt-dlp "https://www.youtube.com/watch?v=HzMls6WtTlQ"


In [ ]:
video = "/content/Traffic video for vehicle detection, classification and tracking I video 2 [HzMls6WtTlQ].mkv"

model3.predict(video, save=True)

In [ ]:
model3.predict(video, save=True)

## ✅ Recap

- **Detection** = find objects **and** their locations (box + label + confidence).
- **YOLO** looks at the whole image once → fast enough for real-time; **NMS/IoU** clean up duplicate boxes.
- Run a pretrained YOLO in **two lines**; `.plot()` draws the boxes.
- Read detections from `results[0].boxes` → `.cls`, `.conf`, `.xyxy`.
- **Filter and count** by looping; restrict classes with the `classes` argument.
- The same call runs on images, folders, **video**, and webcams.

**Next (Lab):** run YOLO on several images, parse and visualise the results yourself, filter by confidence, and build a small object counter. Open `YOLO_Session1_Lab.ipynb`.

**Session 2 preview:** train YOLO on **your own classes** (things COCO doesn't know), and tour real CV applications.